# Lesson 16: Longitudinal Data and Time-Varying Treatments

## Opening Story: HIV Treatment Timing

When should HIV patients start antiretroviral therapy? Starting too early might cause unnecessary side effects; starting too late might allow the virus to damage the immune system. The optimal timing depends on how treatment effects change over time and how patient characteristics evolve.

This is a time-varying treatment problem: the treatment decision at each time point depends on the patient's current and past history, and the treatment itself changes the patient's trajectory.

---

## Learning Objectives

By the end of this lesson, you should be able to:

1. Explain the challenges of time-varying treatments
2. Define marginal structural models
3. Implement inverse probability of treatment weighting (IPTW)
4. Conduct g-computation
5. Handle time-varying confounding

---

## 16.1 Time-Varying Confounding

### The Problem

When treatment affects future covariates, and those covariates affect future treatment and outcomes, we have time-varying confounding. Standard methods like regression adjustment or matching fail because:

1. Conditioning on post-treatment variables introduces bias
2. Not conditioning leaves confounding unaddressed

### Example

```
Treatment → Blood Pressure → Future Treatment
    ↓                              ↓
Outcome ←←←←←←←←←←←←←←←←←←←←←←←←
```

Blood pressure is affected by treatment and affects future treatment decisions. Conditioning on blood pressure blocks part of the treatment effect.

---

## 16.2 Marginal Structural Models

### The Framework

Model the marginal (population-averaged) potential outcomes:

$$E[Y^{\bar{a}}] = g(\bar{a}; \psi)$$

where $\bar{a}$ is a treatment history and $\psi$ are parameters.

### IPTW Estimation

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 1000
n_time = 5

# Generate longitudinal data
data = []
for i in range(n):
    # Initial covariates
    L0 = np.random.normal(0, 1)
    
    for t in range(n_time):
        # Treatment at time t
        prob_treat = 1 / (1 + np.exp(-(-0.5 + 0.5 * L0)))
        A_t = np.random.binomial(1, prob_treat)
        
        # Outcome at time t
        Y_t = 0.5 * L0 + 1.0 * A_t + np.random.normal(0, 0.5)
        
        data.append({
            'id': i,
            'time': t,
            'L': L0,
            'A': A_t,
            'Y': Y_t
        })
        
        # Update covariates
        L0 = L0 + 0.3 * A_t + np.random.normal(0, 0.2)

df = pd.DataFrame(data)

# IPTW weights
def calculate_iptw_weights(df):
    weights = np.ones(df['id'].nunique())
    
    for t in range(n_time):
        t_data = df[df['time'] == t]
        
        # Model treatment given history
        from sklearn.linear_model import LogisticRegression
        
        if t == 0:
            X = t_data[['L']].values
        else:
            # Include past treatment and covariates
            past_data = df[df['time'] < t].groupby('id').last().reset_index()
            X = t_data[['L']].merge(past_data[['id', 'A']], on='id')[['L', 'A']].values
        
        A = t_data['A'].values
        
        # Fit treatment model
        model = LogisticRegression()
        model.fit(X, A)
        prob = model.predict_proba(X)[:, 1]
        
        # Update weights
        w = np.where(A == 1, 1/prob, 1/(1-prob))
        unique_ids = t_data['id'].values
        weights[unique_ids] *= w
    
    return weights

# Calculate stabilized weights
def calculate_stabilized_weights(df):
    weights = np.ones(df['id'].nunique())
    
    for t in range(n_time):
        t_data = df[df['time'] == t]
        
        # Numerator: marginal probability of treatment
        prob_marginal = t_data['A'].mean()
        
        # Denominator: conditional probability
        from sklearn.linear_model import LogisticRegression
        
        if t == 0:
            X = t_data[['L']].values
        else:
            past_data = df[df['time'] < t].groupby('id').last().reset_index()
            X = t_data[['L']].merge(past_data[['id', 'A']], on='id')[['L', 'A']].values
        
        A = t_data['A'].values
        model = LogisticRegression()
        model.fit(X, A)
        prob_conditional = model.predict_proba(X)[:, 1]
        
        # Stabilized weight
        w = np.where(A == 1, prob_marginal/prob_conditional, 
                     (1-prob_marginal)/(1-prob_conditional))
        
        unique_ids = t_data['id'].values
        weights[unique_ids] *= w
    
    return weights

---

## 16.3 G-computation

An alternative to IPTW that models the conditional expectation of outcomes.

---

## 16.4 Common Mistakes

1. **Conditioning on post-treatment variables**: Use IPTW or g-computation
2. **Ignoring time-varying confounding**: Standard methods fail
3. **Model misspecification**: Validate treatment and outcome models
4. **Extreme weights**: Use trimming or stabilized weights

---

## 16.5 Knowledge Check

### Multiple Choice

1. **Time-varying confounding occurs when:**
   A) Treatment affects future covariates
   B) Covariates affect future treatment
   C) Both A and B
   D) Neither

2. **IPTW handles time-varying confounding by:**
   A) Conditioning on covariates
   B) Weighting by inverse probability of treatment
   C) Randomizing treatment
   D) Ignoring confounding

3. **Stabilized weights:**
   A) Reduce variance
   B) Increase bias
   C) Have no effect
   D) Are always necessary

4. **G-computation:**
   A) Is always better than IPTW
   B) Requires correct outcome model
   C) Doesn't need any models
   D) Is only for experiments

5. **Post-treatment variables:**
   A) Should always be controlled
   B) Should never be controlled
   C) Should be controlled only sometimes
   D) Don't exist

### Short Answer

6. **Explain why standard regression fails with time-varying confounding.**

7. **How do IPTW weights account for time-varying confounding?**

8. **What are the advantages of stabilized weights?**

9. **When might g-computation be preferred over IPTW?**

10. **Give an example of a time-varying treatment problem.**

---

## 16.6 Summary

1. **Time-varying confounding** requires special methods
2. **IPTW** weights observations by treatment probability
3. **Stabilized weights** reduce variance
4. **G-computation** models the outcome directly
5. **Sequential ignorability** is the key assumption

---

## 16.7 Further Reading

- Robins, J.M., Hernán, M.A., & Brumback, B. (2000). "Marginal Structural Models and Causal Inference in Epidemiology." *Epidemiology*.
- Hernán, M.A. & Robins, J.M. (2020). *Causal Inference: What If*. Chapman & Hall/CRC.